In [40]:
import random
import pandas as pd
from Bio import SeqIO

In [41]:
""" 
Import Data
"""

# format: bed narrowP, with columns:
cols = [
    "chrom",
    "start",
    "end",
    "peakName",
    "score",
    "strand",
    "signalValue",
    "pValue",
    "qValue",
    "summit",
    "transcriptionFactor",
]

df = pd.read_csv(
    "combined_binding_sites.bed",
    sep="\t", # tab separated
    header=None,
    names=cols,
)

# print(len(df))
df = df.dropna()
df = df[~((df['transcriptionFactor'] == 'ERROR') | (df['transcriptionFactor'] == 'UNKNOWN') | (df['transcriptionFactor'] == 'OTHER'))]
# print(len(df))

df = df.astype({
    "start": "int64",
    "end": "int64",
    "score": "int64",
    "summit": "int64",
    "signalValue": "float64",
    "pValue": "float64",
    "qValue": "float64"
})

df.head()

,chrom,start,end,peakName,score,strand,signalValue,pValue,qValue,summit,transcriptionFactor
0,chr7,4641789,4642324,.,1000,.,450.29518,-1.0,4.34848,324,MYC
1,chr17,77088541,77088861,.,1000,.,440.88340,-1.0,4.34848,158,MYC
2,chr5,171388402,171388744,.,1000,.,413.32234,-1.0,4.34848,192,MYC
3,chr3,131502748,131503082,.,1000,.,395.15057,-1.0,4.34848,166,MYC
4,chr16,4271171,4271541,.,1000,.,390.50998,-1.0,4.34848,176,MYC


In [42]:
""" 
Filter Data by Quality Metrics:

https://gander.wustl.edu/FAQ/FAQformat.html

qValue - Measurement of statistical significance using false discovery rate (-log10(q-value)). Use -1 if no qValue is assigned.
higher = stronger score

signalValue - Measurement of overall (usually, average) enrichment for the region.
higher = stronger binding evidence
"""

df = df[
    (df["qValue"] >= 5) &
    (df["signalValue"] > 10)
]

print("Rows in Dataframe:", len(df))

Rows in Dataframe: 1057125


In [43]:
"""
Import hg38 Human Genome:
"""

fasta_file = "hg38.fa"
genome = SeqIO.to_dict(SeqIO.parse(fasta_file, "fasta"))

In [44]:
"""
Retrieve Binding Sites From Genome:
"""

binding_site_sequences = []

for i, (idx, row) in enumerate(df.iterrows()):
    chrom = row["chrom"]
    start = row["start"]
    end = row["end"]
    strand = row["strand"]
    transcriptionFactor = row["transcriptionFactor"]

    # solely using sequences up to 256 nucleotides in length
    if end-start > 256: 
        continue
    
    # get the nucleotide sequence and converts to upper for consistency
    seq = genome[chrom].seq[start:end].upper()  # hg38 is 0-based

    # inspection print
    if i == 0:
        print("Inspects First Line Values From DF:")
        print("chrom:", chrom)
        print("start:", start)
        print("end:", end)
        print("strand:", strand)
        print("seq:", seq)
    
    # reverse complement if on negative strand
    if strand == "-":
        seq = seq.reverse_complement().upper()

    # 1 = binding site label
    binding_site_sequences.append([1, str(seq), transcriptionFactor])

In [45]:
# first 3 extracted binding site sequences
print("Binding Site Sequences:", len(binding_site_sequences))
print("First 3 Extracted Binding Sites Sequences:")
binding_site_sequences[:3]

Binding Site Sequences: 683638
First 3 Extracted Binding Sites Sequences:


[[1,
  'CTACAAATGACTAAAAGGCACCTCTTCTTTAAGACTTCCGTCTATCATAAAATCATCACACTCTACTGATTTTTGTTAGAACAAGGCAATTTACTGCCATCTGCTGGAAATGTAACATACACAAATCCGCAATGTCTGTAATGTGAGTGGTAACTGCCGGCAATGGGCACCATTGTGCAATGCACAACCTGCACAACTGTACAGAGCAGCTCTGAGTGCTGATGGTAAGACCAGAGGAGCAACAGAG',
  'CTCF'],
 [1,
  'CATCCAACAACAGCAAAAGACATTCTCTGGCTCACATTCATGGGGATAGACCACATTGTGAGCCATAAAAATAGTATGCTAATTAAGCTTTACCAGGGAGTGTTTAACATAGGGTAGACTCTGGCGCCCTCACATGGTCCTCATGTAGGATCTGATTTGGTTTTCCCTGCTAGGTAAAGGTGCAAGATAGAAAGAAGGGTAATTCCAATTTGCTTTCTGTGCTTTGTGATGTATTCTGGTTTACATATGCTCGGAT',
  'CTCF'],
 [1,
  'AGAAAACAAGAAAGTCAGAAAGCCATGGATAAGGAGTGAGCTGGGTAAAAATTGCTTGAAGCAAACAAACAAACAAAATCCCAAGAGGTGTGTATGCATGTCAGCAAGTCATCCACAATCTGGGCAGCAGGTGGCACAAGTAACCTAGATTTCTTAGGCCACTGCAGAAGGAAGCCGCAGAGGTAAGGATTTGGAGAAACTGGGAAGCTGGGAGCATCATCCCAAGCCACTCTTTTGCCTGCCTGATTGCAGTACA',
  'CTCF']]

In [46]:
# min, max and average binding site lengths:

min_sequence_length = min((len(seq[1]) for seq in binding_site_sequences))
print("Minimum Sequence Length:", min_sequence_length)

max_sequence_length = max((len(seq[1]) for seq in binding_site_sequences))
print("Max Sequence Length:", max_sequence_length)

avg_length_binding_site_sequences = int(sum(len(seq[1]) for seq in binding_site_sequences) / len(binding_site_sequences))
print("Average Binding Site Length:", avg_length_binding_site_sequences)

Minimum Sequence Length: 50
Max Sequence Length: 256
Average Binding Site Length: 177


In [47]:
"""
Background DNA Sequence Extraction:
"""

primary_chromosomes = ["chr1", "chr2", "chr3", "chr4", "chr5", "chr6", "chr7", 
                       "chr8", "chr9", "chr10", "chr11", "chr12", "chr13", 
                       "chr14", "chr15", "chr16", "chr17", "chr18", "chr19", 
                       "chr20", "chr21", "chr22", "chrX", "chrY"
                      ]

background_sequences = []
tFactorBackground = "Background"

# number of background sequences to generate = number of binding site sequences
num_background = len(binding_site_sequences)

# background sequence length = average length of binding site sequences
seq_avg_length = int(sum(len(seq[1]) for seq in binding_site_sequences) / len(binding_site_sequences))
seq_min_length = min((len(seq[1]) for seq in binding_site_sequences))
seq_max_length = max((len(seq[1]) for seq in binding_site_sequences))

# generate random sequences
for _ in range(num_background):
    # pick a random primary chromosome
    chrom = random.choice(primary_chromosomes)
    
    seq_record = genome[chrom]
    chrom_len = len(seq_record.seq)
    
    # start = random.randint(0, chrom_len - avg_length)
    
    # pick a random start position
    start = random.randint(0, chrom_len - seq_max_length)
    random_length = random.randint(seq_min_length, seq_max_length)

    end = start + random_length
    
    # extract sequence and uppercase
    random_seq = seq_record.seq[start:end].upper()
    
    # append as background sequence (label 0)
    background_sequences.append([0, str(random_seq), tFactorBackground])

In [48]:
# first 3 extracted backgroun sequences
print("Background Sequences:", len(background_sequences))
print("First 3 Extracted Background Sequences:")
background_sequences[:3]

Background Sequences: 683638
First 3 Extracted Background Sequences:


[[0,
  'AATAAAACATTAAATGTAACCTTATTAGATGTAATAAACGCAACATTCTGGGTTTCCACCTTTTACCTTGTAAATCTTTCCCTGACCATGAGAGAGAGAGAGAGAGAGAGCTGCTATAGTTGCTAAGAATTCATGCTCTGTGTTCAGACAAGAGCTCATATCCCACTT',
  'Background'],
 [0,
  'TTTGTTTGCTTTGTATTTTGTGACCTTCTTAGGCAGAGATAAGCCGGAAGGAGATAGTAGAGGGAAGGTCTTTATTTCTTTGTATCTCAGCCATATTCAAGCTCTAC',
  'Background'],
 [0,
  'GTTTAAATGGAGCTAATTAATATATGCGCCACCTCACATACTTATTATTGTCTTGTGATGAGAACTTTTAATATTTACTCCCTTAGTGATTTTTAAGTATACAATACATTGTTGTTAAGTACAGTCACCATGTTGTACAATAGAGCTCTTTCAGTCATTCCTCCTGTCTGGCTGAAAATTTGTGTCCTCTGACCATTATCTCCCCATTTCCTCTTCTCCCTAGCCC',
  'Background']]

In [49]:
"""
Padding Sequences Prep for Transformer:
"""

# concatenate data
all_sequences = binding_site_sequences + background_sequences

# find max length sequence
max_length = max(len(seq[1]) for seq in all_sequences)
print("Padding Sequences to Length:", max_length)

padded_sequences = []

for label, seq, tFactor in all_sequences:
    # pad with N
    if len(seq) < max_length:
        seq_padded = seq + ("N" * (max_length - len(seq))) 
        
    padded_sequences.append([label, seq_padded, tFactor])

Padding Sequences to Length: 256


In [50]:
avg_length_padded_sequences = int(sum(len(seq[1]) for seq in padded_sequences) / len(padded_sequences))
print("max_length:", max_length)
print("Average Padded Sequence Length:", avg_length_padded_sequences)

max_length: 256
Average Padded Sequence Length: 256


In [51]:
"""
Export Binding Sites to CSV 
"""

seq_df = pd.DataFrame(padded_sequences, columns=["binding_site_label", "sequence", "transcription_factor_label"])

# moving sampling upstream to reduce file size:
yy1_sample = seq_df[seq_df["transcription_factor_label"] == "YY1"].sample(n=5000, random_state=42)
ctcf_sample = seq_df[seq_df["transcription_factor_label"] == "CTCF"].sample(n=5000, random_state=42)
max_sample = seq_df[seq_df["transcription_factor_label"] == "MAX"].sample(n=5000, random_state=42)
background_sample = seq_df[seq_df["transcription_factor_label"] == "Background"].sample(n=5000, random_state=42)

final_df = pd.concat([yy1_sample, ctcf_sample, max_sample, background_sample], ignore_index=True)

final_df.to_csv("All_Sequences.csv", index=False)